In [3]:
import pandas as pd
import numpy as np

In [4]:
rmp_df = pd.read_csv('fake_rmp.csv', index_col = 0)
rmp_df.head()

,Dr. Gerber,Dr. Johnson,Dr. Craig,Dr. Zhu,Dr. Wang,Dr. Murphy,Dr. Sun,Dr. Eagan,Dr. Ness,Dr. Sudyanti,Dr. Zhang,Dr. Wasab,Dr. Diallo,Dr. Hernandez,Dr. King
Student_1,NaN,3.0,NaN,5.0,4.0,4.0,5.0,2.0,NaN,4.0,2.0,4.0,5.0,2.0,3.0
Student_2,NaN,4.0,5.0,5.0,5.0,4.0,4.0,5.0,NaN,4.0,4.0,3.0,5.0,1.0,4.0
Student_3,4.0,4.0,2.0,4.0,3.0,5.0,2.0,4.0,4.0,5.0,2.0,3.0,3.0,5.0,3.0
Student_4,2.0,5.0,5.0,5.0,5.0,2.0,4.0,3.0,5.0,4.0,5.0,4.0,4.0,4.0,5.0
Student_5,3.0,NaN,4.0,NaN,NaN,4.0,5.0,2.0,5.0,3.0,5.0,5.0,2.0,4.0,4.0


## Part 2(a)

In [19]:
def user_collab_filter(df, target_user, similarity='Cosine', k=5):
    # check if the user is in the dataframe
    if target_user not in df.index:
        print(f"Error: {target_user} not found in the dataset.")
        return None

    # check if the user has missing ratings
    if not df.loc[target_user].isna().any():
        print(f"Error: {target_user} has no missing ratings.")
        return None

    # compute each user's average rating in the original dataframe
    user_means = df.mean(axis=1, skipna=True)

    df_avg = df.copy()
    # compute the mean of each row and fill NaN values with the mean
    for idx, row in df_avg.iterrows():
        row_mean = row.mean()
        if pd.isna(row_mean):
            df_avg.loc[idx] = row.fillna(0)
        else:
            df_avg.loc[idx] = row.fillna(row_mean)

    # center each row
    df_centered = df_avg.copy()
    for idx, row in df_centered.iterrows():
        row_mean = row.mean()
        df_centered.loc[idx] -= row_mean

    # find similarity between each user
    similarity_series = pd.Series(index=df.index, dtype=float)
    target_user_row = df_centered.loc[target_user]
    x = target_user_row.to_numpy()
    for idx, row in df_centered.iterrows():
        y = row.to_numpy()
        if similarity == 'Cosine':
            denom = np.linalg.norm(x) * np.linalg.norm(y)
            similarity_score = 0.0 if denom == 0 else np.dot(x, y) / denom
        elif similarity == 'L2':
            similarity_score = -np.linalg.norm(x - y)
        else:
            print(f"Error: {similarity} is not a valid similarity. Please use Cosine or L2.")
            return None
        similarity_series.loc[idx] = similarity_score

    # drop the user's similarity score to themselves
    similarity_series = similarity_series.drop(target_user)

    # min-max scale the similarity series
    similarity_min_max = (similarity_series - similarity_series.min()) / (similarity_series.max() - similarity_series.min())

    # find the k most similar users
    similarity_kmost = similarity_min_max.nlargest(k)
    users_kmost = similarity_kmost.index

    # find empty ratings for the target user
    empty_item_list = df.columns[df.loc[target_user].isna()].tolist()

    # predict ratings using k most similar users
    predicted_rating = {}
    for item in empty_item_list:
        similar_ratings = df.loc[users_kmost, item]
        similar_ratings = similar_ratings.fillna(user_means.loc[users_kmost])

        num = (similarity_kmost * similar_ratings).sum()
        denom = similarity_kmost.sum()
        
        if denom == 0:
            predicted_rating[item] = float(np.nan)
        else:
            predicted_rating[item] = float(num / denom)

    return predicted_rating

## Part 2(b)

In [20]:
user_collab_filter(rmp_df, "Student_5", similarity='Cosine', k=3)

{'Dr. Johnson': 4.14962596753536,
 'Dr. Zhu': 3.6693060979782794,
 'Dr. Wang': 3.6347751229366834}

In [21]:
user_collab_filter(rmp_df, "Student_3", similarity='Cosine', k=3)

Error: Student_3 has no missing ratings.


## Part 2(c)

In [24]:
print(f'Student 1 (cosine): {user_collab_filter(rmp_df, "Student_1", similarity='Cosine', k=5)['Dr. Gerber']}')
print(f'Student 1 (l2): {user_collab_filter(rmp_df, "Student_1", similarity='L2', k=5)['Dr. Gerber']}')

Student 1 (cosine): 4.396127956485075
Student 1 (l2): 3.826632123057846


Based on the computed ratings above, Student 1 would most likely enjoy Dr. Gerber's class. The student would likely give Dr. Gerber a round a 4/5, a solid rating. I would recommend this student to take Dr. Gerber.

In [26]:
print(f'Student 3 (cosine): {user_collab_filter(rmp_df, "Student_2", similarity='Cosine', k=5)['Dr. Gerber']}')
print(f'Student 3 (l2): {user_collab_filter(rmp_df, "Student_2", similarity='L2', k=5)['Dr. Gerber']}')

Student 3 (cosine): 4.251769420851984
Student 3 (l2): 4.56269731203369


Based on the computed ratings above, Student 2 would very likely enjoy Dr. Gerber's class. The student would likely give Dr. Gerber a round a 4.5/5, a strong rating. I would strongly recommend this student to take Dr. Gerber.

In [27]:
print(f'Student 13 (cosine): {user_collab_filter(rmp_df, "Student_13", similarity='Cosine', k=5)['Dr. Gerber']}')
print(f'Student 13 (l2): {user_collab_filter(rmp_df, "Student_13", similarity='L2', k=5)['Dr. Gerber']}')

Student 13 (cosine): 3.4335028612620593
Student 13 (l2): 4.174035557408685


Based on the computed ratings above, Student 13 would likely enjoy Dr. Gerber's class. The student would likely give Dr. Gerber a round a 3.75/5, a decent rating. I would recommend this student to take Dr. Gerber.

## Part 4(a)

In [ ]:
def item_collab_filter(df, target_user, similarity='Cosine', k=5):
    # check if the user is in the dataframe
    if target_user not in df.index:
        print(f"Error: {target_user} not found in the dataset.")
        return None

    # check if the user has missing ratings
    if not df.loc[target_user].isna().any():
        print(f"Error: {target_user} has no missing ratings.")
        return None

    # compute each item's average rating in the original dataframe
    item_means = df.mean(axis=0, skipna=True)

    df_centered = df.copy()
    # center each column
    for col in df_centered.columns:
        col_mean = df_centered[col].mean()
        df_centered[col] = df_centered[col] - col_mean

    # find empty ratings for the target user
    empty_item_list = df.columns[df.loc[target_user].isna()].tolist()

    # predict ratings using k most similar items
    predicted_rating = {}
    for item in empty_item_list:
        # find similarity between each item
        similarity_series = pd.Series(index=df.columns, dtype=float)
        x_full = df_centered[item]

        for col in df_centered.columns:
            y_full = df_centered[col]

            # use only co-rated users for item-item similarity
            corated_mask = x_full.notna() & y_full.notna()
            x = x_full[corated_mask].to_numpy()
            y = y_full[corated_mask].to_numpy()

            if len(x) == 0:
                similarity_score = 0.0
            elif similarity == 'Cosine':
                denom = np.linalg.norm(x) * np.linalg.norm(y)
                similarity_score = 0.0 if denom == 0 else np.dot(x, y) / denom
            elif similarity == 'L2':
                similarity_score = -np.linalg.norm(x - y)
            else:
                print(f"Error: {similarity} is not a valid similarity. Please use Cosine or L2.")
                return None

            similarity_series.loc[col] = similarity_score

        # drop the item's similarity score to itself
        similarity_series = similarity_series.drop(item)

        # min-max scale the similarity series
        similarity_min_max = (similarity_series - similarity_series.min()) / (similarity_series.max() - similarity_series.min())

        # find the k most similar items
        similarity_kmost = similarity_min_max.nlargest(k)
        users_kmost = similarity_kmost.index

        similar_ratings = df.loc[target_user, users_kmost]
        similar_ratings = similar_ratings.fillna(item_means.loc[users_kmost])

        num = (similarity_kmost * similar_ratings).sum()
        denom = similarity_kmost.sum()

        if denom == 0:
            predicted_rating[item] = float(np.nan)
        else:
            predicted_rating[item] = float(num / denom)

    return predicted_rating

In [28]:
def collab_filter(filter_type, df, target_user, similarity='Cosine', k=5):
    if filter_type == 'User':
        return user_collab_filter(df, target_user, similarity, k)
    elif filter_type == 'Item':
        return item_collab_filter(df, target_user, similarity, k)
    else:
        print(f"Error: {filter_type} not found a valid filter. Please use User or Item.")
        return None